In [1]:
import requests
import pandas as pd
import textstat

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True
)

model.eval()

c:\Users\anush\anaconda3main\envs\steering\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 453/453 [00:03<00:00, 121.81it/s, Materializing param=model.layers.31.self_attn.v_proj.weight]
Some parameters are on the meta device because they were offloaded to the cpu.


PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (dense): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_layernorm): LayerNorm((2560,), eps=1

In [3]:
import json

with open("steering_pairs.json", "r") as f:
    steering_pairs = json.load(f)

print("Total pairs:", len(steering_pairs))

Total pairs: 26


In [4]:
import numpy as np

def get_representation(text, layer=16):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.hidden_states[layer]  # (batch, seq_len, dim)

    # Mean pooling
    pooled = hidden_states.mean(dim=1)

    return pooled.squeeze().cpu().numpy()

In [5]:
easy_reps = []
hard_reps = []

for idx, item in enumerate(steering_pairs):
    easy_reps.append(get_representation(item["easy"], layer=16))
    hard_reps.append(get_representation(item["hard"], layer=16))

    if idx % 5 == 0:
        torch.cuda.empty_cache()
        print(f"Processed {idx} samples")

easy_reps = np.vstack(easy_reps)
hard_reps = np.vstack(hard_reps)

print("Done. Shape:", easy_reps.shape)

Processed 0 samples
Processed 5 samples
Processed 10 samples
Processed 15 samples
Processed 20 samples
Processed 25 samples
Done. Shape: (26, 2560)


In [6]:
mu_easy = easy_reps.mean(axis=0)
mu_hard = hard_reps.mean(axis=0)

# IMPORTANT: direction = easy - hard
v_steering = mu_easy - mu_hard

# Normalize
v_steering = v_steering / np.linalg.norm(v_steering)

print("Vector dimension:", v_steering.shape)

Vector dimension: (2560,)


In [7]:
proj_easy = easy_reps @ v_steering
proj_hard = hard_reps @ v_steering

print("Easy mean:", proj_easy.mean())
print("Hard mean:", proj_hard.mean())

Easy mean: 34.5
Hard mean: 9.664


In [8]:
def cohens_d(a, b):
    return (np.mean(a) - np.mean(b)) / np.sqrt(
        (np.var(a) + np.var(b)) / 2
    )

d_proj = cohens_d(proj_easy, proj_hard)
print("Cohen's d (projection space):", d_proj)

Cohen's d (projection space): 5.746


In [9]:
v_torch = torch.tensor(v_steering, dtype=torch.float16).to("cuda")
print(v_torch.shape)  # should be (2560,)

torch.Size([2560])


In [10]:
np.save("v_steering.npy", v_steering)
np.save("mu_easy.npy", mu_easy)
np.save("mu_hard.npy", mu_hard)

print("Saved steering vectors.")

Saved steering vectors.
